# 🤖 AIOS Google Colab LLM Coding Server (Qwen2.5-Coder AWQ)

Запускает защищённый OpenAI-compatible vLLM endpoint. Повторный recovery переиспользует установленные пакеты, кеш Hugging Face и уже загруженную модель. Предпочтительный tunnel — Tailscale Funnel/Serve; при отсутствии Tailscale credentials остаётся явный Cloudflare quick-tunnel fallback. Bearer и Tailscale secrets не должны сохраняться в notebook.


In [ ]:
# === ЯЧЕЙКА 1: Условная установка vLLM и tunnel binary ===
import importlib.util, os, pathlib, shutil, subprocess, sys, urllib.request
provider = os.environ.get("COLAB_TUNNEL_PROVIDER", "auto").lower()
if provider == "auto":
    provider = "tailscale" if os.environ.get("TAILSCALE_AUTH_KEY") and os.environ.get("COLAB_LLM_PUBLIC_URL") else "quick"
if importlib.util.find_spec("vllm") is None:
    print("STAGE install:vllm")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vllm"])
else:
    print("STAGE install:vllm cached")
if pathlib.Path("/content/drive/MyDrive").exists():
    cache = pathlib.Path("/content/drive/MyDrive/AIOS/huggingface_cache")
    cache.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache)
if provider == "tailscale" and shutil.which("tailscale") is None:
    subprocess.check_call("curl -fsSL https://tailscale.com/install.sh | sh", shell=True)
elif provider == "quick" and not pathlib.Path("/usr/local/bin/cloudflared").exists():
    target = pathlib.Path("/tmp/cloudflared.new")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", target)
    target.chmod(0o755)
    os.replace(target, "/usr/local/bin/cloudflared")
print("✅ STAGE install:ready provider=" + provider)


In [ ]:
# === ЯЧЕЙКА 2: Защищённый vLLM OpenAI API на GPU ===
import os, pathlib, subprocess, time, requests, torch
API_KEY = os.environ.get("COLAB_LLM_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError("Inject COLAB_LLM_API_KEY at runtime; never save it in the notebook")
if not torch.cuda.is_available():
    raise RuntimeError("T4 GPU не подключён")
headers = {"Authorization": "Bearer " + API_KEY}
try:
    existing = requests.get("http://127.0.0.1:8000/v1/models", headers=headers, timeout=3)
except Exception:
    existing = None
if existing is not None and existing.status_code == 200:
    print("✅ STAGE model:reused colab/qwen2.5-coder")
else:
    subprocess.run("pkill -f 'vllm.entrypoints.openai.api_server' || true", shell=True)
    log = open("/tmp/aios_vllm.log", "w")
    process = subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ",
        "--served-model-name", "colab/qwen2.5-coder",
        "--quantization", "awq", "--dtype", "half", "--port", "8000",
        "--max-model-len", "4096", "--gpu-memory-utilization", "0.90",
        "--api-key", API_KEY,
    ], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
    for attempt in range(240):
        if process.poll() is not None:
            raise RuntimeError(pathlib.Path("/tmp/aios_vllm.log").read_text(errors="replace")[-3000:])
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", headers=headers, timeout=3).status_code == 200:
                print("✅ vLLM API готов: colab/qwen2.5-coder")
                break
        except Exception:
            pass
        time.sleep(2)
    else:
        raise TimeoutError("vLLM model startup exceeded 8 minutes")


In [ ]:
# === ЯЧЕЙКА 3: Защищённый tunnel (Tailscale preferred) ===
import os, pathlib, re, subprocess, time
provider = os.environ.get("COLAB_TUNNEL_PROVIDER", "auto").lower()
if provider == "auto":
    provider = "tailscale" if os.environ.get("TAILSCALE_AUTH_KEY") and os.environ.get("COLAB_LLM_PUBLIC_URL") else "quick"
if provider == "tailscale":
    auth_key = os.environ.get("TAILSCALE_AUTH_KEY", "").strip()
    public_url = os.environ.get("COLAB_LLM_PUBLIC_URL", "").strip().rstrip("/")
    if not auth_key or not public_url:
        raise RuntimeError("Tailscale runtime credentials/config are missing")
    socket = "/tmp/aios-tailscaled.sock"
    daemon = subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--socket=" + socket, "--state=/tmp/aios-tailscaled.state"])
    for _ in range(30):
        if pathlib.Path(socket).exists(): break
        time.sleep(1)
    subprocess.check_call(["tailscale", "--socket=" + socket, "up", "--reset", "--authkey=" + auth_key, "--hostname=aios-colab-llm", "--accept-routes=false"])
    subprocess.check_call(["tailscale", "--socket=" + socket, "funnel", "--bg", "8000"])
    endpoint = public_url + ("" if public_url.endswith("/v1") else "/v1")
else:
    subprocess.run(["pkill", "-f", "cloudflared tunnel --url http://127.0.0.1:8000"], check=False)
    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    endpoint = ""
    deadline = time.time() + 90
    while time.time() < deadline:
        line = tunnel.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line or "")
        if match:
            endpoint = match.group(0) + "/v1"
            break
    if not endpoint:
        raise RuntimeError("Quick tunnel URL не получен")
print("COLAB_LLM_URL=" + endpoint)
print("🔐 Tunnel and Bearer API ready")
